# 🚀 AIC 2026 — End-to-End Pipeline

### Cấu hình máy
| Thành phần | Chi tiết |
|---|---|
| **GPU** | AMD Radeon RX 5500M (4GB VRAM) — DirectML |
| **CPU** | AMD |
| **Python** | 3.12 + PyTorch + torch-directml |

### Pipeline 6 bước
1. Giải nén dữ liệu BTC
2. Tạo Metadata & Trích xuất Transcript âm thanh (Whisper)
3. **Huấn luyện LoRA CLIP** (GPU AMD — batch=4, accum=8)
4. **Trích xuất Feature Vectors** (GPU AMD DirectML — inference siêu nhanh)
5. Đẩy Vectors lên Qdrant Vector Database
6. Thử nghiệm tìm kiếm

## 📦 Bước 0: Cài đặt thư viện

In [ ]:
!pip install -r requirements.txt

In [ ]:
# Kiểm tra GPU AMD DirectML
import torch
import torch_directml

print(f"PyTorch: {torch.__version__}")
print(f"DirectML available: {torch_directml.is_available()}")
print(f"GPU device: {torch_directml.device()}")
print(f"GPU name: {torch_directml.device_name(0)}")

## 📂 Bước 1: Giải nén dữ liệu BTC
Tự động giải nén các file `.zip` trong `ZIP/` vào đúng cấu trúc `data/`.

In [ ]:
# Xem trước các file sẽ giải nén
!python scripts/extract_btc_data.py --dry-run

In [ ]:
# Giải nén thực tế
!python scripts/extract_btc_data.py

## 📄 Bước 2: Import Metadata & Transcript
Quét keyframes, map frame_id, tích hợp lời thoại âm thanh từ Whisper vào `data/index/metadata.jsonl`.

In [ ]:
!python scripts/import_btc_data.py --with-transcript

## 🧠 Bước 3: Huấn luyện LoRA Fine-Tuning cho CLIP

Fine-tune CLIP ViT-B/32 bằng LoRA adapters (~0.28% tham số, ~1.7MB).

### 🖥️ Chạy trên GPU AMD DirectML
- **batch-size = 4** (vừa VRAM 4GB của AMD Radeon RX 5500M)
- **accum-steps = 8** (Gradient Accumulation → effective batch = 4 × 8 = 32)
- **Yêu cầu:** Đã tăng Windows TDR Timeout lên 60s và **restart máy**

### Thông số
| Thông số | Giá trị | Ý nghĩa |
|---|---|---|
| `--epochs 3` | 3 vòng | 66,744 lượt keyframe |
| `--batch-size 4` | 4 ảnh/lần | Vừa VRAM 4GB GPU AMD |
| `--accum-steps 8` | Tích lũy 8 bước | Effective batch = 32 |

### ⚠️ Nếu GPU bị crash (TDR reset)
Chạy lệnh sau trong PowerShell (Admin) rồi **restart máy**:
```powershell
reg add \"HKLM\\SYSTEM\\CurrentControlSet\\Control\\GraphicsDrivers\" /v TdrDelay /t REG_DWORD /d 60 /f
```
Hoặc fallback CPU: `--batch-size 32 --accum-steps 1 --device cpu`

In [ ]:
# Huấn luyện LoRA CLIP trên GPU AMD DirectML
# (Cần restart máy sau khi tăng TDR Timeout)
!python scripts/train_lora_clip.py --epochs 3 --batch-size 4 --accum-steps 8

In [ ]:
# [TÙY CHỌN] Học tiếp từ checkpoint cũ
# !python scripts/train_lora_clip.py --epochs 3 --batch-size 4 --accum-steps 8 --resume

In [ ]:
# [FALLBACK] Nếu GPU vẫn crash → chạy trên CPU
# !python scripts/train_lora_clip.py --device cpu --epochs 3 --batch-size 32

## 🎨 Bước 4: Trích xuất Feature Vectors bằng LoRA-CLIP

Chiết xuất vector đặc trưng 512d cho toàn bộ keyframes bằng model đã fine-tune.

### ✅ Bước này chạy trên GPU AMD DirectML (siêu nhanh)
- Forward-only inference, không cần backward pass
- GPU AMD Radeon RX 5500M xử lý batch 128 ảnh cùng lúc
- Có cơ chế skip: keyframe đã extract sẽ bị bỏ qua

### ⚠️ Nếu đã extract từ trước khi train LoRA
Cần **xóa clip-features cũ** rồi extract lại để dùng mô hình mới.

In [ ]:
# [TÙY CHỌN] Xóa clip-features cũ nếu vừa train LoRA xong
import shutil
from pathlib import Path

clip_features_dir = Path("data/clip-features")
if clip_features_dir.exists():
    count = sum(1 for _ in clip_features_dir.rglob("*.npy"))
    print(f"Tìm thấy {count} file .npy cũ trong data/clip-features/")
    confirm = input("Xóa toàn bộ để extract lại? (y/n): ")
    if confirm.lower() == 'y':
        shutil.rmtree(clip_features_dir)
        clip_features_dir.mkdir(parents=True, exist_ok=True)
        print("Đã xóa. Sẵn sàng extract lại.")
    else:
        print("Giữ nguyên. Script sẽ skip các file đã tồn tại.")

In [ ]:
# Trích xuất CLIP features trên GPU AMD DirectML
!python scripts/extract_clip_features.py

## 🚀 Bước 5: Đẩy Vectors lên Qdrant Vector Database
Lưu trữ vector và payload metadata lên Qdrant Remote/Local để tìm kiếm similarity.

In [ ]:
!python backend/embedding/push_to_remote.py --recreate

## 🔍 Bước 6: Thử nghiệm Tìm kiếm
Kiểm tra tìm kiếm câu truy vấn mẫu.

In [ ]:
from backend.embedding.clip_encoder import encode_text_raw
import numpy as np

query = "a photo of a tree"
vec = encode_text_raw(query)
print(f"Query: '{query}' -> Vector shape: {vec.shape}, norm: {np.linalg.norm(vec):.4f}")

In [ ]:
# Tìm kiếm trên Qdrant
from qdrant_client import QdrantClient
from backend.config import QDRANT_URL, QDRANT_API_KEY, QDRANT_COLLECTION

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

results = client.search(
    collection_name=QDRANT_COLLECTION,
    query_vector=vec.tolist(),
    limit=10,
)

print(f"\n🔍 Top 10 kết quả cho: '{query}'\n")
for i, hit in enumerate(results, 1):
    p = hit.payload
    print(f"  #{i:02d} | Score: {hit.score:.4f} | {p.get('video_id','')} | Frame: {p.get('frame_id','')} | {p.get('pts_time',0):.1f}s")

---
### 📊 Tóm tắt kiến trúc GPU AMD

| Bước | Thiết bị | Cấu hình |
|---|---|---|
| Training LoRA | **GPU AMD DirectML** | batch=4, accum=8 (cần TDR 60s + restart) |
| Feature Extraction | **GPU AMD DirectML** | batch=128, forward-only |
| Qdrant Push | **CPU** | I/O network |
| Search Query | **GPU AMD DirectML** | Encode text nhanh |